# 03 — Predictive Modeling with Province-Disjoint Group Cross-Validation

Notebook ini mengevaluasi kemampuan model untuk memprediksi jumlah lulusan pada
**provinsi yang tidak pernah dilihat saat pelatihan**.

Dua skema validasi dibandingkan:

1. **GroupKFold berdasarkan provinsi** — estimasi utama generalisasi regional.
2. **Random KFold** — kontrol metodologis untuk mengukur optimisme akibat
   provinsi yang sama muncul pada train dan test.

Model dipilih **hanya** berdasarkan performa GroupKFold. Random CV tidak digunakan
untuk memilih model.

## Model benchmark

- exposure-rate baseline;
- Negative Binomial NB2 dengan offset mahasiswa terdaftar;
- PoissonRegressor pada graduate-output rate;
- TweedieRegressor pada graduate-output rate;
- HistGradientBoosting dengan Poisson loss pada rate;
- Random Forest pada `log1p(jumlah_lulusan)`;
- Extra Trees pada `log1p(jumlah_lulusan)`.

Seluruh preprocessing dipasang hanya pada data train di setiap fold. Provinsi dan
kabupaten/kota tidak digunakan sebagai prediktor, sehingga model tidak dapat
menghafal identitas wilayah.

> Hasil bersifat prediktif-ekologis. Notebook ini tidak mengestimasi peluang
> individual mahasiswa untuk lulus dan tidak membuktikan hubungan kausal.

In [1]:
from __future__ import annotations

import json
import math
import platform
import random
import re
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels.api as sm
from IPython.display import display
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_poisson_deviance,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_SEED = 42
N_SPLITS = 5
BOOTSTRAP_ITERATIONS = 500
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

print('Python      :', sys.version.split()[0])
print('Pandas      :', pd.__version__)
print('NumPy       :', np.__version__)
print('SciPy       :', scipy.__version__)
print('Scikit-learn:', sklearn.__version__)
print('Platform    :', platform.platform())

Python      : 3.13.5
Pandas      : 2.2.3
NumPy       : 2.3.5
SciPy       : 1.17.0
Scikit-learn: 1.8.0
Platform    : Linux-6.12.13-x86_64-with-glibc2.41


In [2]:
CURRENT_DIR = Path.cwd().resolve()
WORKBOOK_NAME = 'Dataset_Gabungan_Kemdiktisaintek_2025.xlsx'


def candidate_roots(start: Path) -> list[Path]:
    roots=[]
    def add(p):
        try: p=p.resolve()
        except OSError: return
        if p not in roots: roots.append(p)
    add(start)
    for p in start.parents: add(p)
    try:
        for child in start.iterdir():
            if child.is_dir(): add(child)
    except Exception:
        pass
    for base in list(roots):
        for name in [
            'pt_education_research_project',
            'pt_education_research_project_00_01_02',
            'pt_education_research_project_00_01_02_03',
        ]:
            add(base/name)
    return roots

PROJECT_ROOT=None
for candidate in candidate_roots(CURRENT_DIR):
    if (candidate/'data'/'processed'/WORKBOOK_NAME).exists():
        PROJECT_ROOT=candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Folder proyek lengkap tidak ditemukan.')

TABLE_DIR=PROJECT_ROOT/'outputs'/'tables'
FIGURE_DIR=PROJECT_ROOT/'outputs'/'figures'
MODEL_DIR=PROJECT_ROOT/'outputs'/'models'
for d in [TABLE_DIR, FIGURE_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

MODEL_DATA_FILE=TABLE_DIR/'02_modeling_dataset.csv'
READINESS_02_FILE=TABLE_DIR/'02_readiness_checklist.csv'
assert MODEL_DATA_FILE.exists(), 'Jalankan Notebook 02 terlebih dahulu.'
if READINESS_02_FILE.exists():
    ready=pd.read_csv(READINESS_02_FILE)
    assert ready['status'].eq('PASS').all(), 'Notebook 02 belum PASS.'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('MODEL_DATA :', MODEL_DATA_FILE)

PROJECT_ROOT: /mnt/data/work_project03/pt_education_research_project
MODEL_DATA : /mnt/data/work_project03/pt_education_research_project/outputs/tables/02_modeling_dataset.csv


## 1. Dataset analitik dan fitur

Identitas provinsi/kabupaten hanya dipakai untuk grouping dan audit, bukan sebagai fitur model.

In [3]:
df=pd.read_csv(MODEL_DATA_FILE)

required=[
    'provinsi_pt','kabupaten_pt','jk','nm_jenj_didik',
    'jumlah_lulusan','jumlah_terdaftar','active_share','mbkm_share',
    'log_pt','pt_high_share','prof_share','cert_share'
]
missing=sorted(set(required)-set(df.columns))
assert not missing, f'Kolom hilang: {missing}'
assert df['jumlah_lulusan'].ge(0).all()
assert df['jumlah_terdaftar'].gt(0).all()
assert df['provinsi_pt'].nunique() >= N_SPLITS

for col in ['provinsi_pt','kabupaten_pt','jk','nm_jenj_didik']:
    df[col]=df[col].astype('string').str.normalize('NFKC').str.replace(r'\s+',' ',regex=True).str.strip()

df['log_enrolled']=np.log1p(df['jumlah_terdaftar'])
df['row_id']=np.arange(len(df))

CATEGORICAL=['jk','nm_jenj_didik']
NUMERIC_RATE=['active_share','mbkm_share','log_pt','pt_high_share','prof_share','cert_share']
NUMERIC_COUNT=NUMERIC_RATE+['log_enrolled']
TARGET='jumlah_lulusan'
EXPOSURE='jumlah_terdaftar'
GROUP='provinsi_pt'
ID_COLS=['row_id','provinsi_pt','kabupaten_pt','jk','nm_jenj_didik']

assert not df.duplicated(['provinsi_pt','kabupaten_pt','jk','nm_jenj_didik']).any()

summary=pd.DataFrame([{
    'rows':len(df), 'provinces':df[GROUP].nunique(),
    'districts':df['kabupaten_pt'].nunique(),
    'total_graduates':df[TARGET].sum(),
    'total_registered':df[EXPOSURE].sum(),
    'minimum_target':df[TARGET].min(),
    'maximum_target':df[TARGET].max(),
}])
summary.to_csv(TABLE_DIR/'03_data_summary.csv',index=False)
display(summary)

,rows,provinces,districts,total_graduates,total_registered,minimum_target,maximum_target
0,2567,34,396,2544731,"10,715,358.000000",1,63535


## 2. Preprocessing dan model

Hyperparameter ditetapkan sebelum evaluasi. Fokus notebook adalah perbandingan protokol validasi, bukan pencarian hyperparameter yang agresif.

In [4]:
def dense_onehot(drop=None):
    return OneHotEncoder(handle_unknown='ignore', drop=drop, sparse_output=False)


def linear_preprocessor():
    return ColumnTransformer([
        ('num',Pipeline([
            ('imputer',SimpleImputer(strategy='median')),
            ('scaler',StandardScaler()),
        ]),NUMERIC_RATE),
        ('cat',Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('onehot',dense_onehot(drop='first')),
        ]),CATEGORICAL),
    ],sparse_threshold=0)


def tree_rate_preprocessor():
    return ColumnTransformer([
        ('num',SimpleImputer(strategy='median'),NUMERIC_RATE),
        ('cat',Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('onehot',dense_onehot()),
        ]),CATEGORICAL),
    ],sparse_threshold=0)


def tree_count_preprocessor():
    return ColumnTransformer([
        ('num',SimpleImputer(strategy='median'),NUMERIC_COUNT),
        ('cat',Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('onehot',dense_onehot()),
        ]),CATEGORICAL),
    ],sparse_threshold=0)

MODEL_NAMES=[
    'ExposureRateBaseline',
    'NB2_Offset',
    'PoissonRate',
    'TweedieRate',
    'HistGBPoissonRate',
    'RandomForestLogCount',
    'ExtraTreesLogCount',
]


def fit_bundle(model_name: str, train: pd.DataFrame):
    y=train[TARGET].to_numpy(float)
    exposure=train[EXPOSURE].to_numpy(float)
    X=train[CATEGORICAL+NUMERIC_COUNT]

    if model_name=='ExposureRateBaseline':
        return {'kind':'baseline','rate':float(y.sum()/exposure.sum())}

    if model_name=='NB2_Offset':
        pre=linear_preprocessor()
        design=pre.fit_transform(train[CATEGORICAL+NUMERIC_RATE])
        design=sm.add_constant(design,has_constant='add')
        result=NegativeBinomial(
            y, design, offset=np.log(exposure), loglike_method='nb2'
        ).fit(disp=False,maxiter=400)
        if not result.mle_retvals.get('converged',False):
            raise RuntimeError('NB2 tidak konvergen.')
        return {'kind':'nb2','preprocessor':pre,'result':result}

    if model_name in {'PoissonRate','TweedieRate','HistGBPoissonRate'}:
        rate=y/exposure*1000.0
        weights=exposure/exposure.mean()
        if model_name=='PoissonRate':
            pipe=Pipeline([
                ('preprocessor',linear_preprocessor()),
                ('model',PoissonRegressor(alpha=0.1,max_iter=300,solver='newton-cholesky')),
            ])
        elif model_name=='TweedieRate':
            pipe=Pipeline([
                ('preprocessor',linear_preprocessor()),
                ('model',TweedieRegressor(power=1.5,alpha=0.1,link='log',max_iter=300,solver='newton-cholesky')),
            ])
        else:
            pipe=Pipeline([
                ('preprocessor',tree_rate_preprocessor()),
                ('model',HistGradientBoostingRegressor(
                    loss='poisson',learning_rate=0.05,max_iter=200,
                    max_leaf_nodes=31,min_samples_leaf=20,
                    l2_regularization=1.0,random_state=RANDOM_SEED,
                )),
            ])
        pipe.fit(X,rate,model__sample_weight=weights)
        return {'kind':'rate','pipeline':pipe}

    if model_name in {'RandomForestLogCount','ExtraTreesLogCount'}:
        if model_name=='RandomForestLogCount':
            estimator=RandomForestRegressor(
                n_estimators=150,min_samples_leaf=2,max_features=0.8,
                n_jobs=1,random_state=RANDOM_SEED,
            )
        else:
            estimator=ExtraTreesRegressor(
                n_estimators=150,min_samples_leaf=2,max_features=1.0,
                n_jobs=1,random_state=RANDOM_SEED,
            )
        pipe=Pipeline([
            ('preprocessor',tree_count_preprocessor()),
            ('model',estimator),
        ])
        pipe.fit(X,np.log1p(y))
        return {'kind':'log_count','pipeline':pipe}

    raise KeyError(model_name)


def predict_bundle(bundle, test: pd.DataFrame) -> np.ndarray:
    exposure=test[EXPOSURE].to_numpy(float)
    X=test[CATEGORICAL+NUMERIC_COUNT]
    kind=bundle['kind']
    if kind=='baseline':
        pred=bundle['rate']*exposure
    elif kind=='nb2':
        design=bundle['preprocessor'].transform(test[CATEGORICAL+NUMERIC_RATE])
        design=sm.add_constant(design,has_constant='add')
        pred=bundle['result'].predict(exog=design,offset=np.log(exposure))
    elif kind=='rate':
        rate=bundle['pipeline'].predict(X)
        pred=rate*exposure/1000.0
    elif kind=='log_count':
        pred=np.expm1(bundle['pipeline'].predict(X))
    else:
        raise KeyError(kind)
    return np.clip(np.asarray(pred,float),1e-9,None)


def metric_dict(y,p):
    y=np.asarray(y,float); p=np.clip(np.asarray(p,float),1e-9,None)
    return {
        'mae':float(mean_absolute_error(y,p)),
        'rmse':float(mean_squared_error(y,p)**0.5),
        'rmsle':float(mean_squared_log_error(y,p)**0.5),
        'mean_poisson_deviance':float(mean_poisson_deviance(y,p)),
        'log_r2':float(r2_score(np.log1p(y),np.log1p(p))),
        'spearman_r':float(stats.spearmanr(y,p).statistic),
        'observed_total':float(y.sum()),
        'predicted_total':float(p.sum()),
        'predicted_to_observed_ratio':float(p.sum()/y.sum()),
    }


def save_figure(fig,name):
    path=FIGURE_DIR/name
    fig.savefig(path,dpi=300,bbox_inches='tight')
    plt.close(fig)
    return path

## 3. Split dan audit kebocoran wilayah

In [5]:
splitters={
    'GroupKFold_Province':GroupKFold(n_splits=N_SPLITS),
    'RandomKFold':KFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_SEED),
}

splits={}
leakage_rows=[]
assignment_rows=[]
for scheme,splitter in splitters.items():
    if scheme=='GroupKFold_Province':
        iterator=splitter.split(df,df[TARGET],groups=df[GROUP])
    else:
        iterator=splitter.split(df,df[TARGET])
    scheme_splits=[]
    for fold,(train_idx,test_idx) in enumerate(iterator,1):
        train_prov=set(df.iloc[train_idx][GROUP])
        test_prov=set(df.iloc[test_idx][GROUP])
        overlap=train_prov & test_prov
        leakage_rows.append({
            'scheme':scheme,'fold':fold,
            'train_rows':len(train_idx),'test_rows':len(test_idx),
            'train_provinces':len(train_prov),'test_provinces':len(test_prov),
            'overlapping_provinces':len(overlap),
            'test_graduates':float(df.iloc[test_idx][TARGET].sum()),
            'test_registered':float(df.iloc[test_idx][EXPOSURE].sum()),
        })
        for idx in test_idx:
            assignment_rows.append({'scheme':scheme,'row_id':int(df.iloc[idx]['row_id']),'fold':fold})
        scheme_splits.append((train_idx,test_idx))
    splits[scheme]=scheme_splits

leakage_audit=pd.DataFrame(leakage_rows)
fold_assignments=pd.DataFrame(assignment_rows)
leakage_audit.to_csv(TABLE_DIR/'03_split_leakage_audit.csv',index=False)
fold_assignments.to_csv(TABLE_DIR/'03_fold_assignments.csv',index=False)
display(leakage_audit)

assert leakage_audit.loc[leakage_audit['scheme'].eq('GroupKFold_Province'),'overlapping_provinces'].eq(0).all()
assert leakage_audit.loc[leakage_audit['scheme'].eq('RandomKFold'),'overlapping_provinces'].gt(0).all()
print('PASS — GroupKFold province-disjoint; random CV mengandung overlap provinsi.')

,scheme,fold,train_rows,test_rows,train_provinces,test_provinces,overlapping_provinces,test_graduates,test_registered
0,GroupKFold_Province,1,2052,515,28,6,0,"495,147.000000","1,792,974.000000"
1,GroupKFold_Province,2,2047,520,27,7,0,"539,655.000000","3,308,844.000000"
2,GroupKFold_Province,3,2062,505,28,6,0,"580,932.000000","2,122,210.000000"
3,GroupKFold_Province,4,2062,505,27,7,0,"559,024.000000","2,089,637.000000"
4,GroupKFold_Province,5,2045,522,26,8,0,"369,973.000000","1,401,693.000000"
5,RandomKFold,1,2053,514,34,34,34,"479,772.000000","1,617,598.000000"
6,RandomKFold,2,2053,514,34,34,34,"544,633.000000","1,956,163.000000"
7,RandomKFold,3,2054,513,34,34,34,"527,157.000000","1,840,397.000000"
8,RandomKFold,4,2054,513,34,34,34,"456,318.000000","2,516,565.000000"
9,RandomKFold,5,2054,513,34,33,33,"536,851.000000","2,784,635.000000"


PASS — GroupKFold province-disjoint; random CV mengandung overlap provinsi.


## 4. Out-of-fold evaluation

In [6]:
oof_rows=[]
fold_metric_rows=[]
fit_time_rows=[]

for scheme,scheme_splits in splits.items():
    print('\n===',scheme,'===')
    for fold,(train_idx,test_idx) in enumerate(scheme_splits,1):
        train=df.iloc[train_idx].copy()
        test=df.iloc[test_idx].copy()
        print(f'Fold {fold}: train={len(train):,}, test={len(test):,}')
        for model_name in MODEL_NAMES:
            start=time.perf_counter()
            bundle=fit_bundle(model_name,train)
            pred=predict_bundle(bundle,test)
            elapsed=time.perf_counter()-start
            assert np.isfinite(pred).all() and (pred>0).all()
            metrics=metric_dict(test[TARGET],pred)
            fold_metric_rows.append({
                'scheme':scheme,'fold':fold,'model':model_name,
                **metrics,'fit_predict_seconds':elapsed,
            })
            fit_time_rows.append({'scheme':scheme,'fold':fold,'model':model_name,'seconds':elapsed})
            temp=test[ID_COLS+[TARGET,EXPOSURE]].copy()
            temp['scheme']=scheme
            temp['fold']=fold
            temp['model']=model_name
            temp['predicted_lulusan']=pred
            temp['absolute_log_error']=np.abs(np.log1p(temp[TARGET])-np.log1p(pred))
            oof_rows.append(temp)

oof=pd.concat(oof_rows,ignore_index=True)
fold_metrics=pd.DataFrame(fold_metric_rows)
fit_times=pd.DataFrame(fit_time_rows)

oof.to_csv(TABLE_DIR/'03_oof_predictions.csv',index=False)
fold_metrics.to_csv(TABLE_DIR/'03_fold_metrics.csv',index=False)
fit_times.to_csv(TABLE_DIR/'03_fit_times.csv',index=False)

expected=len(df)*len(MODEL_NAMES)*len(splitters)
assert len(oof)==expected, (len(oof),expected)
counts=oof.groupby(['scheme','model'])['row_id'].nunique()
assert counts.eq(len(df)).all()
print('OOF rows:',f'{len(oof):,}')


=== GroupKFold_Province ===
Fold 1: train=2,052, test=515


Fold 2: train=2,047, test=520


Fold 3: train=2,062, test=505


Fold 4: train=2,062, test=505


Fold 5: train=2,045, test=522


...

Fold 2: train=2,053, test=514


Fold 3: train=2,054, test=513


Fold 4: train=2,054, test=513


Fold 5: train=2,054, test=513


OOF rows: 35,938


## 5. Metrik strata dan agregat provinsi

Pemilihan model menggunakan **province-level RMSLE pada GroupKFold**, sehingga setiap provinsi memperoleh bobot yang sama.

In [7]:
metric_rows=[]
province_rows=[]
province_metric_rows=[]

for (scheme,model),part in oof.groupby(['scheme','model']):
    metric_rows.append({'scheme':scheme,'model':model,'level':'stratum',**metric_dict(part[TARGET],part['predicted_lulusan'])})
    prov=(part.groupby('provinsi_pt',as_index=False)
          .agg(observed_lulusan=(TARGET,'sum'),predicted_lulusan=('predicted_lulusan','sum'),
               registered=(EXPOSURE,'sum'),n_strata=(TARGET,'size')))
    prov['scheme']=scheme; prov['model']=model
    prov['absolute_log_error']=np.abs(np.log1p(prov['observed_lulusan'])-np.log1p(prov['predicted_lulusan']))
    prov['residual']=prov['observed_lulusan']-prov['predicted_lulusan']
    province_rows.append(prov)
    pm=metric_dict(prov['observed_lulusan'],prov['predicted_lulusan'])
    province_metric_rows.append({'scheme':scheme,'model':model,'level':'province','n_provinces':len(prov),**pm})

metrics=pd.concat([pd.DataFrame(metric_rows),pd.DataFrame(province_metric_rows)],ignore_index=True,sort=False)
province_predictions=pd.concat(province_rows,ignore_index=True)
metrics.to_csv(TABLE_DIR/'03_model_metrics.csv',index=False)
province_predictions.to_csv(TABLE_DIR/'03_province_predictions.csv',index=False)

group_province=metrics[(metrics['scheme']=='GroupKFold_Province')&(metrics['level']=='province')].sort_values('rmsle')
BEST_MODEL=group_province.iloc[0]['model']

model_ranking=group_province.copy()
model_ranking['rank_rmsle']=model_ranking['rmsle'].rank(method='min')
model_ranking.to_csv(TABLE_DIR/'03_groupcv_model_ranking.csv',index=False)

display(model_ranking[['model','rmsle','mae','mean_poisson_deviance','log_r2','predicted_to_observed_ratio','rank_rmsle']])
print('BEST MODEL (province GroupCV RMSLE):',BEST_MODEL)

,model,rmsle,mae,mean_poisson_deviance,log_r2,predicted_to_observed_ratio,rank_rmsle
20,TweedieRate,0.225527,"11,260.176799","3,395.540036",0.964301,1.034721,1.000000
18,PoissonRate,0.226900,"11,402.150580","3,751.978121",0.963865,1.036140,2.000000
15,ExtraTreesLogCount,0.238169,"9,260.136395","3,176.786039",0.960186,0.904854,3.000000
16,HistGBPoissonRate,0.247037,"10,985.622955","2,385.313618",0.957166,1.054821,4.000000
19,RandomForestLogCount,0.247614,"11,819.489390","3,815.483456",0.956966,0.880812,5.000000
17,NB2_Offset,0.250585,"11,990.647449","3,148.389091",0.955927,1.095490,6.000000
14,ExposureRateBaseline,0.343849,"23,252.607301","14,290.932559",0.917015,1.022100,7.000000


BEST MODEL (province GroupCV RMSLE): TweedieRate


## 6. Generalization gap: random versus province-disjoint CV

In [8]:
wide=(metrics[metrics['level'].eq('province')]
      .pivot(index='model',columns='scheme',values=['rmsle','mae','mean_poisson_deviance','log_r2','predicted_to_observed_ratio']))
wide.columns=['__'.join(col) for col in wide.columns]
wide=wide.reset_index()
wide['rmsle_gap_group_minus_random']=wide['rmsle__GroupKFold_Province']-wide['rmsle__RandomKFold']
wide['rmsle_ratio_group_over_random']=wide['rmsle__GroupKFold_Province']/wide['rmsle__RandomKFold']
wide['mae_gap_group_minus_random']=wide['mae__GroupKFold_Province']-wide['mae__RandomKFold']
wide['log_r2_drop_random_to_group']=wide['log_r2__RandomKFold']-wide['log_r2__GroupKFold_Province']
wide=wide.sort_values('rmsle_gap_group_minus_random',ascending=False)
wide.to_csv(TABLE_DIR/'03_generalization_gap.csv',index=False)
display(wide[['model','rmsle__RandomKFold','rmsle__GroupKFold_Province','rmsle_gap_group_minus_random','rmsle_ratio_group_over_random','log_r2_drop_random_to_group']])

,model,rmsle__RandomKFold,rmsle__GroupKFold_Province,rmsle_gap_group_minus_random,rmsle_ratio_group_over_random,log_r2_drop_random_to_group
2,HistGBPoissonRate,0.125557,0.247037,0.121480,1.967525,0.031769
5,RandomForestLogCount,0.216075,0.247614,0.031539,1.145965,0.010265
1,ExtraTreesLogCount,0.218353,0.238169,0.019816,1.090753,0.006350
4,PoissonRate,0.215894,0.226900,0.011006,1.050979,0.003421
3,NB2_Offset,0.239899,0.250585,0.010687,1.044546,0.003679
0,ExposureRateBaseline,0.334870,0.343849,0.008979,1.026815,0.004278
6,TweedieRate,0.222570,0.225527,0.002956,1.013282,0.000930


## 7. Bootstrap provinsi dan perbandingan berpasangan

In [9]:
rng=np.random.default_rng(RANDOM_SEED)
bootstrap_rows=[]
province_group=province_predictions[province_predictions['scheme'].eq('GroupKFold_Province')]

for model,part in province_group.groupby('model'):
    part=part.reset_index(drop=True)
    values=[]
    for _ in range(BOOTSTRAP_ITERATIONS):
        sample=part.iloc[rng.integers(0,len(part),len(part))]
        m=metric_dict(sample['observed_lulusan'],sample['predicted_lulusan'])
        values.append([m['rmsle'],m['mae'],m['log_r2'],m['predicted_to_observed_ratio']])
    arr=np.asarray(values)
    for j,metric in enumerate(['rmsle','mae','log_r2','predicted_to_observed_ratio']):
        estimate=metric_dict(part['observed_lulusan'],part['predicted_lulusan'])[metric]
        bootstrap_rows.append({
            'model':model,'metric':metric,'estimate':estimate,
            'ci_lower_2_5':float(np.quantile(arr[:,j],0.025)),
            'ci_upper_97_5':float(np.quantile(arr[:,j],0.975)),
            'bootstrap_iterations':BOOTSTRAP_ITERATIONS,
        })
bootstrap_ci=pd.DataFrame(bootstrap_rows)
bootstrap_ci.to_csv(TABLE_DIR/'03_groupcv_province_bootstrap_ci.csv',index=False)
display(bootstrap_ci[(bootstrap_ci['metric']=='rmsle')].sort_values('estimate'))

best_errors=(province_group[province_group['model'].eq(BEST_MODEL)]
             .set_index('provinsi_pt')['absolute_log_error'])
wilcoxon=[]
for model,part in province_group.groupby('model'):
    if model==BEST_MODEL: continue
    other=part.set_index('provinsi_pt')['absolute_log_error'].reindex(best_errors.index)
    diff=other-best_errors
    try:
        test=stats.wilcoxon(diff,alternative='greater',zero_method='wilcox')
        stat,p=float(test.statistic),float(test.pvalue)
    except ValueError:
        stat,p=np.nan,1.0
    wilcoxon.append({'best_model':BEST_MODEL,'comparison_model':model,
                     'mean_error_best':float(best_errors.mean()),'mean_error_comparison':float(other.mean()),
                     'wilcoxon_statistic':stat,'p_value':p})
wilcoxon=pd.DataFrame(wilcoxon)
if len(wilcoxon):
    wilcoxon['p_adjusted_bh']=multipletests(wilcoxon['p_value'],method='fdr_bh')[1]
    wilcoxon['comparison_worse_at_0_05']=wilcoxon['p_adjusted_bh']<0.05
wilcoxon.to_csv(TABLE_DIR/'03_pairwise_wilcoxon_vs_best.csv',index=False)
display(wilcoxon.sort_values('p_adjusted_bh'))

,model,metric,estimate,ci_lower_2_5,ci_upper_97_5,bootstrap_iterations
24,TweedieRate,rmsle,0.225527,0.170612,0.268193,500
16,PoissonRate,rmsle,0.226900,0.175459,0.280251,500
4,ExtraTreesLogCount,rmsle,0.238169,0.162357,0.304566,500
8,HistGBPoissonRate,rmsle,0.247037,0.202908,0.295698,500
20,RandomForestLogCount,rmsle,0.247614,0.169136,0.323093,500
12,NB2_Offset,rmsle,0.250585,0.199082,0.299643,500
0,ExposureRateBaseline,rmsle,0.343849,0.210313,0.480538,500


,best_model,comparison_model,mean_error_best,mean_error_comparison,wilcoxon_statistic,p_value,p_adjusted_bh,comparison_worse_at_0_05
0,TweedieRate,ExposureRateBaseline,0.181415,0.239012,375.000000,0.095281,0.277755,False
2,TweedieRate,HistGBPoissonRate,0.181415,0.208796,362.000000,0.138878,0.277755,False
3,TweedieRate,NB2_Offset,0.181415,0.212439,384.000000,0.071502,0.277755,False
5,TweedieRate,RandomForestLogCount,0.181415,0.185123,306.000000,0.446310,0.669466,False
1,TweedieRate,ExtraTreesLogCount,0.181415,0.163147,228.000000,0.882652,0.882652,False
4,TweedieRate,PoissonRate,0.181415,0.180092,258.000000,0.750553,0.882652,False


## 8. Diagnostik model terbaik pada GroupKFold

In [10]:
best_oof=oof[(oof['scheme']=='GroupKFold_Province')&(oof['model']==BEST_MODEL)].copy()
best_prov=province_predictions[(province_predictions['scheme']=='GroupKFold_Province')&(province_predictions['model']==BEST_MODEL)].copy()

ranked=best_oof['predicted_lulusan'].rank(method='first')
best_oof['prediction_decile']=pd.qcut(ranked,10,labels=False)+1
calibration=(best_oof.groupby('prediction_decile',as_index=False)
             .agg(n=(TARGET,'size'),observed_total=(TARGET,'sum'),predicted_total=('predicted_lulusan','sum'),
                  observed_mean=(TARGET,'mean'),predicted_mean=('predicted_lulusan','mean')))
calibration['observed_to_predicted_ratio']=calibration['observed_total']/calibration['predicted_total']
calibration.to_csv(TABLE_DIR/'03_best_groupcv_calibration_deciles.csv',index=False)

worst=best_prov.sort_values('absolute_log_error',ascending=False).head(15)
worst.to_csv(TABLE_DIR/'03_best_model_worst_provinces.csv',index=False)
display(calibration)
display(worst[['provinsi_pt','observed_lulusan','predicted_lulusan','absolute_log_error','residual']])

,prediction_decile,n,observed_total,predicted_total,observed_mean,predicted_mean,observed_to_predicted_ratio
0,1,257,1765,"1,234.142411",6.867704,4.802111,1.430143
1,2,257,4136,"3,841.107867",16.093385,14.945945,1.076773
2,3,256,8832,"8,151.162760",34.500000,31.840480,1.083526
3,4,257,14753,"14,444.840894",57.404669,56.205607,1.021334
4,5,257,29192,"23,949.946204",113.587549,93.190452,1.218875
5,6,256,43918,"40,706.342048",171.554688,159.009149,1.078898
6,7,257,83071,"73,221.856974",323.233463,284.909949,1.134511
7,8,256,150468,"140,616.704674",587.765625,549.284003,1.070058
8,9,257,335065,"322,037.229646","1,303.754864","1,253.063150",1.040454
9,10,257,1873531,"2,004,884.052151","7,290.003891","7,801.105261",0.934483


,provinsi_pt,observed_lulusan,predicted_lulusan,absolute_log_error,residual
207,Prov. Banten,153271,"276,212.514520",0.588960,"-122,941.514520"
208,Prov. Bengkulu,25023,"16,753.710711",0.401156,"8,269.289289"
235,Prov. Sumatera Barat,86458,"58,227.421440",0.395297,"28,230.578560"
234,Prov. Sulawesi Utara,24974,"16,916.361423",0.389535,"8,057.638577"
224,Prov. Maluku Utara,10471,"7,427.838505",0.343336,"3,043.161495"
223,Prov. Maluku,18492,"13,421.546126",0.320456,"5,070.453874"
236,Prov. Sumatera Selatan,56467,"42,191.902140",0.291422,"14,275.097860"
219,Prov. Kalimantan Timur,26373,"34,883.572924",0.279666,"-8,510.572924"
220,Prov. Kalimantan Utara,4104,"5,374.754636",0.269693,"-1,270.754636"
222,Prov. Lampung,48131,"37,010.012512",0.262732,"11,120.987488"


In [11]:
# Figure 1: RMSLE random vs group.
plot=metrics[metrics['level'].eq('province')].pivot(index='model',columns='scheme',values='rmsle').loc[MODEL_NAMES]
x=np.arange(len(plot)); width=0.36
fig,ax=plt.subplots(figsize=(12,6))
ax.bar(x-width/2,plot['RandomKFold'],width,label='Random KFold')
ax.bar(x+width/2,plot['GroupKFold_Province'],width,label='GroupKFold provinsi')
ax.set_xticks(x); ax.set_xticklabels(plot.index,rotation=35,ha='right')
ax.set_ylabel('Province-level RMSLE')
ax.set_title('Random CV versus Province-Disjoint Group CV')
ax.legend(); ax.grid(axis='y',alpha=0.2)
save_figure(fig,'03_01_random_vs_group_rmsle.png')

# Figure 2: generalization gap.
gap=wide.sort_values('rmsle_gap_group_minus_random')
fig,ax=plt.subplots(figsize=(9,5.5))
ax.barh(gap['model'],gap['rmsle_gap_group_minus_random'])
ax.axvline(0,linestyle='--',linewidth=1)
ax.set_xlabel('GroupCV RMSLE minus RandomCV RMSLE')
ax.set_title('Optimisme Random CV')
ax.grid(axis='x',alpha=0.2)
save_figure(fig,'03_02_generalization_gap.png')

# Figure 3: model ranking under GroupCV.
rank_plot=model_ranking.sort_values('rmsle',ascending=False)
fig,ax=plt.subplots(figsize=(9,5.5))
ax.barh(rank_plot['model'],rank_plot['rmsle'])
ax.set_xlabel('Province-level RMSLE')
ax.set_title('Model Ranking — Province-Disjoint GroupCV')
ax.grid(axis='x',alpha=0.2)
save_figure(fig,'03_03_groupcv_model_ranking.png')

# Figure 4: observed vs predicted province.
fig,ax=plt.subplots(figsize=(7,6.5))
ax.scatter(best_prov['predicted_lulusan'],best_prov['observed_lulusan'],s=35)
lo=min(best_prov['predicted_lulusan'].min(),best_prov['observed_lulusan'].min())
hi=max(best_prov['predicted_lulusan'].max(),best_prov['observed_lulusan'].max())
ax.plot([lo,hi],[lo,hi],linestyle='--')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Prediksi total lulusan provinsi')
ax.set_ylabel('Observasi total lulusan provinsi')
ax.set_title(f'Observed vs Predicted Province — {BEST_MODEL}')
ax.grid(alpha=0.2)
save_figure(fig,'03_04_best_observed_vs_predicted_province.png')

# Figure 5: province residuals.
resplot=best_prov.sort_values('residual')
fig,ax=plt.subplots(figsize=(9,max(7,0.25*len(resplot))))
ax.barh(resplot['provinsi_pt'],resplot['residual'])
ax.axvline(0,linestyle='--',linewidth=1)
ax.set_xlabel('Observed minus predicted graduates')
ax.set_title(f'Residual Provinsi — {BEST_MODEL}')
ax.grid(axis='x',alpha=0.2)
save_figure(fig,'03_05_best_province_residuals.png')

# Figure 6: calibration.
fig,ax=plt.subplots(figsize=(7,5.5))
ax.plot(calibration['predicted_mean'],calibration['observed_mean'],marker='o')
lo=min(calibration['predicted_mean'].min(),calibration['observed_mean'].min())
hi=max(calibration['predicted_mean'].max(),calibration['observed_mean'].max())
ax.plot([lo,hi],[lo,hi],linestyle='--')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Rata-rata prediksi per desil')
ax.set_ylabel('Rata-rata observasi per desil')
ax.set_title(f'GroupCV Calibration — {BEST_MODEL}')
ax.grid(alpha=0.2)
save_figure(fig,'03_06_best_calibration_deciles.png')

# Figure 7: fold RMSLE boxplot.
fm=fold_metrics[fold_metrics['scheme'].eq('GroupKFold_Province')]
arrays=[fm.loc[fm['model'].eq(m),'rmsle'].to_numpy() for m in MODEL_NAMES]
fig,ax=plt.subplots(figsize=(12,6))
ax.boxplot(arrays,tick_labels=MODEL_NAMES,showmeans=True)
ax.set_xticklabels(MODEL_NAMES,rotation=35,ha='right')
ax.set_ylabel('Fold RMSLE')
ax.set_title('Variasi Performa antar GroupKFold')
ax.grid(axis='y',alpha=0.2)
save_figure(fig,'03_07_groupcv_fold_rmsle.png')

# Figure 8: stratum observed/predicted.
fig,ax=plt.subplots(figsize=(7,6.5))
ax.scatter(best_oof['predicted_lulusan'],best_oof[TARGET],alpha=0.3,s=15)
lo=min(best_oof['predicted_lulusan'].min(),best_oof[TARGET].min())
hi=max(best_oof['predicted_lulusan'].max(),best_oof[TARGET].max())
ax.plot([lo,hi],[lo,hi],linestyle='--')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Prediksi jumlah lulusan strata')
ax.set_ylabel('Observasi jumlah lulusan strata')
ax.set_title(f'OOF Stratum Predictions — {BEST_MODEL}')
ax.grid(alpha=0.2)
save_figure(fig,'03_08_best_stratum_observed_vs_predicted.png')

# Figure 9: fit time.
timeplot=fit_times.groupby('model',as_index=False)['seconds'].mean().sort_values('seconds')
fig,ax=plt.subplots(figsize=(9,5.5))
ax.barh(timeplot['model'],timeplot['seconds'])
ax.set_xlabel('Rata-rata fit+predict seconds per fold')
ax.set_title('Biaya Komputasi Model')
ax.grid(axis='x',alpha=0.2)
save_figure(fig,'03_09_model_fit_time.png')

PosixPath('/mnt/data/work_project03/pt_education_research_project/outputs/figures/03_09_model_fit_time.png')

## 9. Fit model terbaik pada seluruh data dan simpan artefak

In [12]:
full_start=time.perf_counter()
full_bundle=fit_bundle(BEST_MODEL,df)
full_seconds=time.perf_counter()-full_start
slug=re.sub(r'[^a-z0-9]+','_',BEST_MODEL.lower()).strip('_')
best_model_path=MODEL_DIR/f'03_best_model_{slug}.joblib'
joblib.dump(full_bundle,best_model_path)

registry={
    'best_model':BEST_MODEL,
    'selection_metric':'province-level RMSLE under GroupKFold by province',
    'training_rows':len(df),
    'training_provinces':df[GROUP].nunique(),
    'features':{'categorical':CATEGORICAL,'numeric_rate':NUMERIC_RATE,'numeric_count':NUMERIC_COUNT},
    'target':TARGET,'exposure':EXPOSURE,'group':GROUP,
    'model_file':str(best_model_path.relative_to(PROJECT_ROOT)),
    'fit_seconds_full_data':full_seconds,
    'random_seed':RANDOM_SEED,
}
with (MODEL_DIR/'03_model_registry.json').open('w',encoding='utf-8') as f:
    json.dump(registry,f,ensure_ascii=False,indent=2)
print(json.dumps(registry,ensure_ascii=False,indent=2))

{
  "best_model": "TweedieRate",
  "selection_metric": "province-level RMSLE under GroupKFold by province",
  "training_rows": 2567,
  "training_provinces": 34,
  "features": {
    "categorical": [
      "jk",
      "nm_jenj_didik"
    ],
    "numeric_rate": [
      "active_share",
      "mbkm_share",
      "log_pt",
      "pt_high_share",
      "prof_share",
      "cert_share"
    ],
    "numeric_count": [
      "active_share",
      "mbkm_share",
      "log_pt",
      "pt_high_share",
      "prof_share",
      "cert_share",
      "log_enrolled"
    ]
  },
  "target": "jumlah_lulusan",
  "exposure": "jumlah_terdaftar",
  "group": "provinsi_pt",
  "model_file": "outputs/models/03_best_model_tweedierate.joblib",
  "fit_seconds_full_data": 0.011202865000086604,
  "random_seed": 42
}


## 10. Ringkasan dan readiness checklist

In [13]:
best_group=metrics[(metrics['scheme']=='GroupKFold_Province')&(metrics['level']=='province')&(metrics['model']==BEST_MODEL)].iloc[0]
best_random=metrics[(metrics['scheme']=='RandomKFold')&(metrics['level']=='province')&(metrics['model']==BEST_MODEL)].iloc[0]
best_ci=bootstrap_ci[(bootstrap_ci['model']==BEST_MODEL)&(bootstrap_ci['metric']=='rmsle')].iloc[0]

summary={
    'best_model':BEST_MODEL,
    'groupcv_province_rmsle':float(best_group['rmsle']),
    'groupcv_province_rmsle_ci_lower':float(best_ci['ci_lower_2_5']),
    'groupcv_province_rmsle_ci_upper':float(best_ci['ci_upper_97_5']),
    'randomcv_province_rmsle':float(best_random['rmsle']),
    'rmsle_generalization_gap':float(best_group['rmsle']-best_random['rmsle']),
    'rmsle_group_over_random_ratio':float(best_group['rmsle']/best_random['rmsle']),
    'groupcv_province_mae':float(best_group['mae']),
    'groupcv_log_r2':float(best_group['log_r2']),
    'groupcv_calibration_ratio':float(best_group['predicted_to_observed_ratio']),
    'models_compared':len(MODEL_NAMES),
    'n_provinces':int(df[GROUP].nunique()),
}
with (TABLE_DIR/'03_summary.json').open('w',encoding='utf-8') as f:
    json.dump(summary,f,ensure_ascii=False,indent=2)

findings=f"""# Ringkasan Hasil Notebook 03

- Evaluasi utama menggunakan province-disjoint GroupKFold pada {df[GROUP].nunique()} provinsi.
- Model terbaik berdasarkan province-level RMSLE adalah **{BEST_MODEL}**.
- GroupCV RMSLE: **{best_group['rmsle']:.4f}** (bootstrap 95% CI {best_ci['ci_lower_2_5']:.4f}–{best_ci['ci_upper_97_5']:.4f}).
- Random CV RMSLE: **{best_random['rmsle']:.4f}**.
- Generalization gap: **{best_group['rmsle']-best_random['rmsle']:.4f}**; GroupCV/RandomCV ratio **{best_group['rmsle']/best_random['rmsle']:.3f}**.
- GroupCV province MAE: **{best_group['mae']:,.2f}** lulusan.
- GroupCV log-R²: **{best_group['log_r2']:.4f}**.
- Predicted/observed total ratio: **{best_group['predicted_to_observed_ratio']:.4f}**.
- Random CV memiliki overlap provinsi pada seluruh fold, sedangkan GroupCV tidak memiliki overlap.
- Hasil random CV hanya dipakai untuk mendemonstrasikan optimisme validasi, bukan memilih model.
"""
(TABLE_DIR/'03_key_findings.md').write_text(findings,encoding='utf-8')
print(findings)

expected_tables=[
 '03_data_summary.csv','03_split_leakage_audit.csv','03_fold_assignments.csv',
 '03_oof_predictions.csv','03_fold_metrics.csv','03_fit_times.csv','03_model_metrics.csv',
 '03_province_predictions.csv','03_groupcv_model_ranking.csv','03_generalization_gap.csv',
 '03_groupcv_province_bootstrap_ci.csv','03_pairwise_wilcoxon_vs_best.csv',
 '03_best_groupcv_calibration_deciles.csv','03_best_model_worst_provinces.csv',
 '03_summary.json','03_key_findings.md'
]
expected_figures=[f'03_0{i}_' for i in range(1,10)]
checks=[
 {'check':'GroupCV tanpa overlap provinsi','status':'PASS' if leakage_audit.loc[leakage_audit.scheme.eq('GroupKFold_Province'),'overlapping_provinces'].eq(0).all() else 'FAIL','evidence':'0 overlap pada seluruh fold'},
 {'check':'OOF lengkap','status':'PASS' if len(oof)==expected else 'FAIL','evidence':f'{len(oof):,}/{expected:,} rows'},
 {'check':'Seluruh model menghasilkan prediksi','status':'PASS' if set(oof.model)==set(MODEL_NAMES) else 'FAIL','evidence':f'{oof.model.nunique()}/{len(MODEL_NAMES)} models'},
 {'check':'Bootstrap selesai','status':'PASS' if len(bootstrap_ci)==len(MODEL_NAMES)*4 else 'FAIL','evidence':f'{BOOTSTRAP_ITERATIONS} iterations'},
 {'check':'Model terbaik tersimpan','status':'PASS' if best_model_path.exists() else 'FAIL','evidence':best_model_path.name},
 {'check':'Tabel lengkap','status':'PASS' if all((TABLE_DIR/x).exists() for x in expected_tables) else 'FAIL','evidence':f'{sum((TABLE_DIR/x).exists() for x in expected_tables)}/{len(expected_tables)}'},
 {'check':'Figur lengkap','status':'PASS' if len(list(FIGURE_DIR.glob('03_*.png')))>=9 else 'FAIL','evidence':f"{len(list(FIGURE_DIR.glob('03_*.png')))}/9"},
]
readiness=pd.DataFrame(checks)
readiness.to_csv(TABLE_DIR/'03_readiness_checklist.csv',index=False)
display(readiness)
assert readiness['status'].eq('PASS').all(), 'Notebook 03 belum PASS.'

manifest=[]
for folder,kind in [(TABLE_DIR,'table'),(FIGURE_DIR,'figure'),(MODEL_DIR,'model')]:
    for path in sorted(folder.glob('03_*')):
        if path.is_file():
            manifest.append({'artifact_type':kind,'filename':path.name,'relative_path':str(path.relative_to(PROJECT_ROOT)),'size_bytes':path.stat().st_size})
pd.DataFrame(manifest).to_csv(TABLE_DIR/'03_output_manifest.csv',index=False)

print('='*76)
print('NOTEBOOK 03 LOLOS.')
print('Evaluasi siap dilanjutkan ke 04_STEM_Regional_Analysis.ipynb')
print('='*76)

# Ringkasan Hasil Notebook 03

- Evaluasi utama menggunakan province-disjoint GroupKFold pada 34 provinsi.
- Model terbaik berdasarkan province-level RMSLE adalah **TweedieRate**.
- GroupCV RMSLE: **0.2255** (bootstrap 95% CI 0.1706–0.2682).
- Random CV RMSLE: **0.2226**.
- Generalization gap: **0.0030**; GroupCV/RandomCV ratio **1.013**.
- GroupCV province MAE: **11,260.18** lulusan.
- GroupCV log-R²: **0.9643**.
- Predicted/observed total ratio: **1.0347**.
- Random CV memiliki overlap provinsi pada seluruh fold, sedangkan GroupCV tidak memiliki overlap.
- Hasil random CV hanya dipakai untuk mendemonstrasikan optimisme validasi, bukan memilih model.



,check,status,evidence
0,GroupCV tanpa overlap provinsi,PASS,0 overlap pada seluruh fold
1,OOF lengkap,PASS,"35,938/35,938 rows"
2,Seluruh model menghasilkan prediksi,PASS,7/7 models
3,Bootstrap selesai,PASS,500 iterations
4,Model terbaik tersimpan,PASS,03_best_model_tweedierate.joblib
5,Tabel lengkap,PASS,16/16
6,Figur lengkap,PASS,10/9


NOTEBOOK 03 LOLOS.
Evaluasi siap dilanjutkan ke 04_STEM_Regional_Analysis.ipynb
